In [1]:
# Cell 1: Setup
import sys
sys.path.append('../src')
import numpy as np
from data_loader import DataLoader
from feature_extraction import FeatureExtractor
from model_training import ModelTrainer

# Cell 2: Load Data and Extract Features (reuse from previous)
loader = DataLoader()
X_train, X_val, X_test, y_train, y_val, y_test = loader.prepare_data()

extractor = FeatureExtractor()


Loading MNIST dataset...
✓ Dataset loaded: 70000 samples, 784 features
✓ Pixels normalized using standard
✓ Data split:
   Train = 49000
   Validation = 7000
   Test = 14000


In [2]:
print("\n=== FEATURE EXTRACTION ===")

# ------------------------------------------------
# Extract HOG Features
# ------------------------------------------------
hog_train = extractor.extract_hog_features(X_train)

hog_val = extractor.extract_hog_features(X_val)

hog_test = extractor.extract_hog_features(X_test)

# ------------------------------------------------
# Apply PCA
# ------------------------------------------------
features_train = extractor.apply_pca(
    hog_train,
    n_components=100,
    fit=True
)

features_val = extractor.apply_pca(
    hog_val,
    fit=False
)

features_test = extractor.apply_pca(
    hog_test,
    fit=False
)

print("\n✓ Feature extraction completed")

print(f"Train Features Shape: {features_train.shape}")
print(f"Validation Features Shape: {features_val.shape}")
print(f"Test Features Shape: {features_test.shape}")




=== FEATURE EXTRACTION ===
Extracting HOG features...
  Processed 5000 images
  Processed 10000 images
  Processed 15000 images
  Processed 20000 images
  Processed 25000 images
  Processed 30000 images
  Processed 35000 images
  Processed 40000 images
  Processed 45000 images
✓ HOG features extracted
  Shape: (49000, 1296)
Extracting HOG features...
  Processed 5000 images
✓ HOG features extracted
  Shape: (7000, 1296)
Extracting HOG features...
  Processed 5000 images
  Processed 10000 images
✓ HOG features extracted
  Shape: (14000, 1296)
✓ PCA applied
  Components: 100
  Explained Variance: 0.7528 (75.28%)

✓ Feature extraction completed
Train Features Shape: (49000, 100)
Validation Features Shape: (7000, 100)
Test Features Shape: (14000, 100)


In [3]:

trainer = ModelTrainer()
models = trainer.train_all_models(features_train, y_train)

print("\n=== HYPERPARAMETER TUNING ===")
best_svm = trainer.hyperparameter_tuning_svm(
    features_train, y_train,
    C_values=[0.1, 1, 10, 100],
    gamma_values=['scale', 0.001, 0.01, 0.1]
)

trainer.save_models('../models/')

print("\n=== VALIDATION ACCURACY ===")
for model_name, model in trainer.models.items():
    val_acc = model.score(features_val, y_val)
    print(f"{model_name}: {val_acc:.4f}")


TRAINING ALL MODELS

=== TRAINING SVM (RBF KERNEL) ===
✓ SVM trained in 5.10s
  Hyperparameters: C=1.0, gamma=scale


AttributeError: 'LinearSVC' object has no attribute 'support_vectors_'